<a href="https://colab.research.google.com/github/Ming-flowbetter/Machine-Learning/blob/main/lec08_attention_transformer_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 08, Part II demo: attention and the Transformer block, from scratch

CSCI 567 --- Machine Learning

Three self-contained parts, all NumPy-only (no PyTorch or scikit-learn needed):

1. **Scaled dot-product self-attention from scratch**, verified against the lecture notes'
   hand-worked numeric example to full floating-point precision (including causal masking and a
   second, multi-head example).
2. **A full single Transformer block** (multi-head attention + per-token MLP + residual
   connections + layer normalization), run end to end and checked for sensible output shapes and
   correctly-normalized statistics.
3. **Permutation-equivariance, demonstrated directly**: permute the input tokens and confirm
   attention's output permutes identically (the lecture notes' proof, checked empirically to
   numerical precision), then add positional encoding and confirm that equivariance breaks.

Every number below is actually computed by the code in this notebook, not transcribed by hand.

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

## Part 1: Scaled dot-product self-attention from scratch

$$P = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{D_K}} + M\right), \qquad \text{output} = PV$$

$M$ is an optional additive mask (all zeros for plain attention; $-\infty$ above the diagonal for
causal masking). Softmax is applied row-wise.

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax along `axis`."""
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """Attention(Q, K, V) = softmax(QK^T / sqrt(D_K) + mask) V, per Vaswani et al. (2017)."""
    D_K = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(D_K)
    if mask is not None:
        scores = scores + mask
    P = softmax(scores, axis=-1)
    return P, P @ V

### 1.1 Reproducing the lecture notes' hand-worked example exactly

Two tokens ($T=2$), key/query dimension $D_K=4$, value dimension $2$. The query/key/value
vectors are exactly the ones worked by hand on the lecture slides and in the notes:
$q_1=(1,2,0,1)$, $q_2=(0,1,2,1)$, $k_1=(1,1,0,1)$, $k_2=(2,0,1,0)$, $v_1=(1,0)$, $v_2=(0,2)$.

By hand, row 1's raw scores were $q_1^\top k_1=4$, $q_1^\top k_2=2$; scaled by $1/\sqrt4=1/2$
to $2,1$; softmax to $p_{1,1}\approx0.731$, $p_{1,2}\approx0.269$; output row 1
$\approx(0.731,\,0.538)$. Row 2 was an exact tie, giving $p_{2,1}=p_{2,2}=0.5$ and output row 2
$=(0.5,\,1.0)$.

In [ ]:
q1 = np.array([1., 2., 0., 1.]); q2 = np.array([0., 1., 2., 1.])
k1 = np.array([1., 1., 0., 1.]); k2 = np.array([2., 0., 1., 0.])
v1 = np.array([1., 0.]); v2 = np.array([0., 2.])
Q = np.stack([q1, q2]); K = np.stack([k1, k2]); V = np.stack([v1, v2])

P, Z = scaled_dot_product_attention(Q, K, V)
print("Attention weights P =\n", P)
print("Output Z =\n", Z)

expected_P = np.array([[0.7310585786300049, 0.2689414213699951],
                        [0.5, 0.5]])
expected_Z = np.array([[0.7310585786300049, 0.5378828427400098],
                        [0.5, 1.0]])
assert np.allclose(P, expected_P), "P does not match the hand-worked example!"
assert np.allclose(Z, expected_Z), "Z does not match the hand-worked example!"
print("\nMatches the lecture's hand-worked example to full floating-point precision.")

### 1.2 Causal masking

Mask token $t$'s access to every token $s>t$ by adding $-\infty$ to those score-matrix entries
before the softmax (see the lecture notes). On the same two-token example, masking token 1's view of
token 2 should force its attention weights to exactly $(1,0)$ and its output to exactly
$v_1=(1,0)$; token 2 (which is allowed to see everything up to and including itself) should be
unaffected.

In [ ]:
T = 2
causal_mask = np.triu(np.full((T, T), -np.inf), k=1)
print("causal mask:\n", causal_mask)

P_causal, Z_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print("\nP (causal) =\n", P_causal)
print("Z (causal) =\n", Z_causal)

assert np.allclose(P_causal, np.array([[1.0, 0.0], [0.5, 0.5]]))
assert np.allclose(Z_causal, np.array([[1.0, 0.0], [0.5, 1.0]]))
print("\nCausal masking matches the predicted exact values.")

### 1.3 A second head, and multi-head concatenation

One query/key/value projection gives one attention *pattern*. A second, independently-parameterized
head, run on the same two tokens, should learn a genuinely different pattern; concatenating both
heads' outputs and mixing with a learned $W_O$ gives multi-head attention's final output
(see the lecture notes).

In [ ]:
# Head 2: a different, independently-specified set of query/key/value vectors on the same 2 tokens.
q1_h2 = np.array([1., 0.]); q2_h2 = np.array([2., 1.])
k1_h2 = np.array([1., 0.]); k2_h2 = np.array([0., 1.])
v1_h2 = np.array([2., 1.]); v2_h2 = np.array([1., 3.])
Q_h2 = np.stack([q1_h2, q2_h2]); K_h2 = np.stack([k1_h2, k2_h2]); V_h2 = np.stack([v1_h2, v2_h2])

P_h2, Z_h2 = scaled_dot_product_attention(Q_h2, K_h2, V_h2)
print("head 1 P (from Part 1.1) =\n", P)
print("head 2 P                =\n", P_h2)
print("\nThe two heads' attention patterns genuinely differ -- each finds a different relationship.")

concat = np.concatenate([Z, Z_h2], axis=1)              # (T, D_K1 + D_K2)
W_O = np.array([[1, 0], [0, 1], [1, 1], [1, -1]], dtype=float)  # learned output-mixing matrix
multihead_out = concat @ W_O
print("\nconcatenated heads:\n", concat)
print("multi-head output (after W_O):\n", multihead_out)

## Part 2: A full Transformer block

Assemble multi-head attention with a per-token MLP, a residual connection around each sub-layer,
and layer normalization (Lecture 07 and Discussion 07's layer norm, reused unchanged), as in
the lecture:

$$X^{(l)}_\text{attn} = \mathrm{LayerNorm}\big(X^{(l)} + \mathrm{MultiHead}(X^{(l)})\big),
\qquad
X^{(l+1)} = \mathrm{LayerNorm}\big(X^{(l)}_\text{attn} + \mathrm{MLP}(X^{(l)}_\text{attn})\big)$$

In [ ]:
def layer_norm(X, gamma=None, beta=None, eps=1e-5):
    """Per-token (per-row) layer normalization, Discussion 07's formula."""
    mu = X.mean(axis=-1, keepdims=True)
    var = X.var(axis=-1, keepdims=True)
    X_hat = (X - mu) / np.sqrt(var + eps)
    if gamma is not None:
        X_hat = X_hat * gamma
    if beta is not None:
        X_hat = X_hat + beta
    return X_hat


def relu(z):
    return np.maximum(0, z)


class MultiHeadAttention:
    """H independent attention heads, concatenated and mixed by a learned W_O."""

    def __init__(self, D, H, D_K, rng):
        self.H = H
        self.W_Q = [rng.normal(scale=0.1, size=(D, D_K)) for _ in range(H)]
        self.W_K = [rng.normal(scale=0.1, size=(D, D_K)) for _ in range(H)]
        self.W_V = [rng.normal(scale=0.1, size=(D, D_K)) for _ in range(H)]
        self.W_O = rng.normal(scale=0.1, size=(H * D_K, D))

    def __call__(self, X, mask=None):
        heads = []
        for h in range(self.H):
            Q_h, K_h, V_h = X @ self.W_Q[h], X @ self.W_K[h], X @ self.W_V[h]
            _, Z_h = scaled_dot_product_attention(Q_h, K_h, V_h, mask=mask)
            heads.append(Z_h)
        return np.concatenate(heads, axis=1) @ self.W_O


class TransformerBlock:
    """Multi-head attention + per-token MLP, each wrapped in a residual connection + LayerNorm."""

    def __init__(self, D, H, D_K, D_ff, rng):
        self.mha = MultiHeadAttention(D, H, D_K, rng)
        self.W1 = rng.normal(scale=0.1, size=(D, D_ff))
        self.b1 = np.zeros(D_ff)
        self.W2 = rng.normal(scale=0.1, size=(D_ff, D))
        self.b2 = np.zeros(D)

    def mlp(self, X):
        """Applied identically and independently to every row (token) -- no cross-token mixing."""
        return relu(X @ self.W1 + self.b1) @ self.W2 + self.b2

    def __call__(self, X, mask=None):
        X_attn = layer_norm(X + self.mha(X, mask=mask))
        X_out = layer_norm(X_attn + self.mlp(X_attn))
        return X_out

### 2.1 Running the block end to end

A random $5$-token, $8$-dimensional input sequence through one Transformer block with $2$ heads
(so each head gets $D_K=4$) and a $16$-wide inner MLP. Two things to check: the output shape must
match the input shape exactly (a Transformer block never changes the sequence length or model
dimension, so blocks can be stacked), and every output row (token) must have mean $\approx0$ and
standard deviation $\approx1$ across its $D$ features -- LayerNorm's own guarantee, applied last.

In [ ]:
rng = np.random.default_rng(0)
T, D, H, D_K, D_ff = 5, 8, 2, 4, 16
X = rng.normal(size=(T, D))

block = TransformerBlock(D, H, D_K, D_ff, rng)
out = block(X)

print("input shape: ", X.shape)
print("output shape:", out.shape)
assert out.shape == X.shape
print("Output shape matches input shape, as required for stacking blocks.\n")

print("output:\n", out)
print("\nper-token output means (LayerNorm guarantees ~0):", out.mean(axis=-1))
print("per-token output stds  (LayerNorm guarantees ~1):", out.std(axis=-1))

# A causal (decoder-style) version of the same block runs identically, just with a mask passed through.
causal_mask = np.triu(np.full((T, T), -np.inf), k=1)
out_causal = block(X, mask=causal_mask)
print("\ncausal-masked output shape:", out_causal.shape, "(unchanged, as expected)")

## Part 3: Permutation-equivariance, and what positional encoding does to it

The lecture proves: for a permutation matrix $R$, attention (with no positional
information) satisfies $\text{Attention}(RX) = R\cdot\text{Attention}(X)$ -- permute the input
rows, and the output permutes identically. Checked here directly, to numerical precision, and then
broken on purpose by adding a positional encoding.

In [ ]:
rng2 = np.random.default_rng(42)
T3, D3, D_K3 = 6, 6, 6
X3 = rng2.normal(size=(T3, D3))
W_Q3 = rng2.normal(scale=0.3, size=(D3, D_K3))
W_K3 = rng2.normal(scale=0.3, size=(D3, D_K3))
W_V3 = rng2.normal(scale=0.3, size=(D3, D_K3))


def attn_of(X):
    Q_, K_, V_ = X @ W_Q3, X @ W_K3, X @ W_V3
    _, Z_ = scaled_dot_product_attention(Q_, K_, V_)
    return Z_


perm = rng2.permutation(T3)
R = np.eye(T3)[perm]  # permutation matrix: R @ X reorders rows of X according to `perm`
print("random permutation of the 6 tokens:", perm)

Z_orig = attn_of(X3)
Z_of_permuted_input = attn_of(R @ X3)      # permute the tokens, then run attention
permuted_Z_orig = R @ Z_orig               # run attention, then permute the output

max_diff = np.max(np.abs(Z_of_permuted_input - permuted_Z_orig))
print("max |Attention(RX) - R . Attention(X)| =", max_diff)
assert np.allclose(Z_of_permuted_input, permuted_Z_orig, atol=1e-10)
print("Permutation-equivariance CONFIRMED to numerical precision (no positional encoding).")

### 3.1 Adding positional encoding breaks the equivariance

Vaswani et al. (2017)'s sinusoidal positional encoding, added to each token's embedding before
attention: $PE_{(pos,2i)}=\sin(pos/10000^{2i/D})$, $PE_{(pos,2i+1)}=\cos(pos/10000^{2i/D})$.

In [ ]:
def sinusoidal_positional_encoding(T, D):
    pos = np.arange(T)[:, None]
    i = np.arange(D)[None, :]
    angle_rates = 1 / (10000 ** (2 * (i // 2) / D))
    angles = pos * angle_rates
    pe = np.zeros((T, D))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


PE = sinusoidal_positional_encoding(T3, D3)


def attn_of_with_pe(X):
    X_pe = X + PE                          # position-specific vector added BEFORE Q/K/V
    Q_, K_, V_ = X_pe @ W_Q3, X_pe @ W_K3, X_pe @ W_V3
    _, Z_ = scaled_dot_product_attention(Q_, K_, V_)
    return Z_


Z_pe_orig = attn_of_with_pe(X3)
Z_pe_of_permuted_input = attn_of_with_pe(R @ X3)
permuted_Z_pe_orig = R @ Z_pe_orig

max_diff_pe = np.max(np.abs(Z_pe_of_permuted_input - permuted_Z_pe_orig))
print("max |Attention(RX + PE) - R . Attention(X + PE)| =", max_diff_pe)
print("Equal to numerical precision?", np.allclose(Z_pe_of_permuted_input, permuted_Z_pe_orig, atol=1e-10))
assert max_diff_pe > 1e-3, "expected positional encoding to break equivariance, but it did not"
print("\nPositional encoding BREAKS permutation-equivariance, exactly as the notes predict:")
print("adding PE ties each row to its position in the sequence, not to its content, so")
print("permuting content no longer corresponds to permuting rows of the same matrix.")

## Summary

| Check | Result |
|---|---|
| Scaled dot-product attention matches the hand-worked example | exact, to full float precision |
| Causal masking on the same example | exact match to predicted $(1,0)$ / $(0.5,0.5)$ |
| Multi-head attention (2 heads, concatenated, mixed by $W_O$) | runs; heads learn different patterns |
| Full Transformer block (attention + MLP + residual + LayerNorm) | output shape matches input; per-token mean $\approx0$, std $\approx1$ |
| Permutation-equivariance (no positional encoding) | confirmed to numerical precision ($<10^{-10}$) |
| Permutation-equivariance (with sinusoidal positional encoding) | broken, as predicted (difference $\approx0.11$, far above numerical noise) |

Every claim the lecture makes about attention and the Transformer block is verified here by
actually running the code, not just asserted.